In [1]:
import pandas as pd
import numpy as np
import os

def create_generalized_features(df):
    # Validate required columns first
    required_cols = ['date', 'team_h', 'team_a', 'h_goals', 'a_goals']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # Sắp xếp theo thời gian
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")
    
    # Tạo các cột kết quả
    df["result"] = np.where(df["h_goals"] > df["a_goals"], 2, 
                   np.where(df["h_goals"] == df["a_goals"], 1, 0))
    df["total_goals"] = df["h_goals"] + df["a_goals"]
    df["over_2.5"] = (df["total_goals"] > 2.5).astype(int)
    
    # Tính toán phong độ (Rolling averages)
    def get_rolling_stats(team_name, group):
        group = group.sort_values("date")
        stats = []
        
        for i, row in group.iterrows():
            is_home = row["team_h"] == team_name
            prefix = "h_" if is_home else "a_"
            opp_prefix = "a_" if is_home else "h_"
            
            stats.append({
                "date": row["date"],
                "goals_for": row[f"{prefix}goals"],
                "goals_against": row[f"{opp_prefix}goals"],
                "xg_for": row.get(f"{prefix}xg", 0),
                "xg_against": row.get(f"{opp_prefix}xg", 0),
                "shots_for": row.get(f"{prefix}shot", 0),
                "shots_against": row.get(f"{opp_prefix}shot", 0),
                "ppda": row.get(f"{prefix}ppda", 0),
                "deep": row.get(f"{prefix}deep", 0)
            })
        
        stats_df = pd.DataFrame(stats)
        rolling = stats_df.set_index("date").rolling(window=5, closed="left").mean().reset_index()
        rolling.columns = ["date"] + [f"roll_{c}" for c in rolling.columns if c != "date"]
        return rolling
    
    all_teams = pd.concat([df["team_h"], df["team_a"]]).unique()
    team_stats = {}
    
    for team in all_teams:
        team_matches = df[(df["team_h"] == team) | (df["team_a"] == team)]
        team_stats[team] = get_rolling_stats(team, team_matches)
    
    features_h, features_a = [], []
    
    for i, row in df.iterrows():
        h_team, a_team, date = row["team_h"], row["team_a"], row["date"]
        
        h_feat_row = team_stats[h_team][team_stats[h_team]["date"] == date]
        a_feat_row = team_stats[a_team][team_stats[a_team]["date"] == date]
        
        if not h_feat_row.empty and not a_feat_row.empty:
            h_feat = h_feat_row.iloc[0].to_dict()
            a_feat = a_feat_row.iloc[0].to_dict()
            del h_feat["date"]
            del a_feat["date"]
            
            features_h.append({f"h_{k}": v for k, v in h_feat.items()})
            features_a.append({f"a_{k}": v for k, v in a_feat.items()})
        else:
            null_dict = {f"roll_{k}": np.nan for k in 
                        ["goals_for", "goals_against", "xg_for", "xg_against", 
                         "shots_for", "shots_against", "ppda", "deep"]}
            features_h.append({f"h_{k}": v for k, v in null_dict.items()})
            features_a.append({f"a_{k}": v for k, v in null_dict.items()})
    
    final_df = pd.concat([df.reset_index(drop=True), 
                          pd.DataFrame(features_h), 
                          pd.DataFrame(features_a)], axis=1)
    
    # --- GENERALIZATION STEP: Relative Features ---
    stats_to_relativize = ["goals_for", "goals_against", "xg_for", "xg_against", 
                           "shots_for", "shots_against", "ppda", "deep"]
    
    for stat in stats_to_relativize:
        final_df[f"rel_{stat}"] = final_df[f"h_roll_{stat}"] - final_df[f"a_roll_{stat}"]
    
    return final_df.dropna()


# Main execution
base_path = "archive/understats/"
leagues = ["EPL", "Bundesliga", "La_Liga", "Ligue_1", "Serie_A", "RFPL"]

all_data = []

for league in leagues:
    path = os.path.join(base_path, league, "match_info.csv")
    
    if not os.path.exists(path):
        print(f"⚠ Warning: {league} file not found at {path}")
        continue
    
    print(f"Processing {league}...")
    
    try:
        # Try semicolon separator first
        df = pd.read_csv(path, sep=";")
        
        # Check if date column exists, if not try comma separator
        if "date" not in df.columns:
            print(f"  Retrying with comma separator...")
            df = pd.read_csv(path, sep=",")
        
        # Verify we have the minimum required columns
        if "date" not in df.columns or "team_h" not in df.columns:
            print(f"  ✗ Error: Missing required columns. Available columns: {df.columns.tolist()}")
            continue
        
        df["league_name"] = league
        
        # Process the dataframe
        processed_df = create_generalized_features(df)
        all_data.append(processed_df)
        
        print(f"  ✓ Successfully processed {len(processed_df)} matches")
        
    except Exception as e:
        print(f"  ✗ Error processing {league}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

# Check if we have any data before concatenating
if len(all_data) == 0:
    print("\n❌ ERROR: No data was successfully processed!")
    print("Please check:")
    print(f"1. The path exists: {base_path}")
    print("2. CSV files exist in subdirectories")
    print("3. CSV files have the correct format and columns")
else:
    global_df = pd.concat(all_data, ignore_index=True)
    global_df.to_csv("global_features.csv", index=False)
    
    print(f"\n✓ Global feature engineering complete!")
    print(f"  Total matches: {len(global_df)}")
    print(f"  Leagues processed: {global_df['league_name'].nunique()}")
    print(f"  Output saved to: global_features.csv")

Processing EPL...
  ✓ Successfully processed 3681 matches
Processing Bundesliga...
  ✓ Successfully processed 2965 matches
Processing La_Liga...
  ✓ Successfully processed 3695 matches
Processing Ligue_1...
  Retrying with comma separator...
  ✓ Successfully processed 3511 matches
Processing Serie_A...
  Retrying with comma separator...
  ✓ Successfully processed 3675 matches
Processing RFPL...
  Retrying with comma separator...
  ✓ Successfully processed 2284 matches

✓ Global feature engineering complete!
  Total matches: 19811
  Leagues processed: 6
  Output saved to: global_features.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from scipy.stats import poisson
import joblib

# 1. Load and Prepare Data
df = pd.read_csv("global_features.csv")

# Define features
rel_features = ["rel_goals_for", "rel_goals_against", "rel_xg_for", "rel_xg_against", "rel_shots_for", "rel_shots_against", "rel_ppda", "rel_deep"]
base_features = ["h_roll_goals_for", "h_roll_goals_against", "h_roll_xg_for", "h_roll_xg_against", "h_roll_shots_for", "h_roll_shots_against", "h_roll_ppda", "h_roll_deep",
                 "a_roll_goals_for", "a_roll_goals_against", "a_roll_xg_for", "a_roll_xg_against", "a_roll_shots_for", "a_roll_shots_against", "a_roll_ppda", "a_roll_deep"]
all_features = rel_features + base_features

X = df[all_features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Train ML Models (RandomForest)
# 1x2 Model
y_1x2 = df["result"]
clf_1x2 = RandomForestClassifier(n_estimators=150, max_depth=15, random_state=42, n_jobs=-1)
clf_1x2.fit(X_scaled, y_1x2)

# Over/Under 2.5 Model
y_ou = df["over_2.5"]
clf_ou = RandomForestClassifier(n_estimators=150, max_depth=15, random_state=42, n_jobs=-1)
clf_ou.fit(X_scaled, y_ou)

# Score Regressors (to get lambda for Poisson)
y_h_goals = df["h_goals"]
y_a_goals = df["a_goals"]
reg_h = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
reg_a = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
reg_h.fit(X_scaled, y_h_goals)
reg_a.fit(X_scaled, y_a_goals)

# 3. Hybrid Logic: Poisson Distribution for Score Probabilities
def get_score_probabilities(lambda_h, lambda_a, max_goals=5):
    """Calculate a matrix of score probabilities using Poisson distribution."""
    prob_matrix = np.outer(poisson.pmf(range(max_goals + 1), lambda_h),
                           poisson.pmf(range(max_goals + 1), lambda_a))
    return prob_matrix

def predict_hybrid(X_input_scaled):
    # ML Predictions
    prob_1x2_ml = clf_1x2.predict_proba(X_input_scaled)[0] # [Away, Draw, Home]
    prob_ou_ml = clf_ou.predict_proba(X_input_scaled)[0]   # [Under, Over]
    
    # Poisson Predictions (using ML predicted goals as lambda)
    lambda_h = reg_h.predict(X_input_scaled)[0]
    lambda_a = reg_a.predict(X_input_scaled)[0]
    
    score_probs = get_score_probabilities(lambda_h, lambda_a)
    
    # Derive 1x2 from Poisson for comparison/hybridization
    prob_home_poisson = np.sum(np.tril(score_probs, -1).T) # Home > Away
    prob_draw_poisson = np.sum(np.diag(score_probs))      # Home == Away
    prob_away_poisson = np.sum(np.triu(score_probs, 1).T)  # Away > Home
    
    # Hybrid 1x2 (Simple average for demonstration)
    hybrid_1x2 = {
        "Home Win": (prob_1x2_ml[2] + prob_home_poisson) / 2,
        "Draw": (prob_1x2_ml[1] + prob_draw_poisson) / 2,
        "Away Win": (prob_1x2_ml[0] + prob_away_poisson) / 2
    }
    
    # Most likely score from Poisson
    max_idx = np.unravel_index(np.argmax(score_probs), score_probs.shape)
    predicted_score = f"{max_idx[0]}-{max_idx[1]}"
    
    return {
        "hybrid_1x2": hybrid_1x2,
        "ml_ou_2.5": {"Under 2.5": prob_ou_ml[0], "Over 2.5": prob_ou_ml[1]},
        "predicted_score": predicted_score,
        "score_matrix": score_probs.tolist()
    }

# 4. Save everything for Web Deployment
model_package = {
    "clf_1x2": clf_1x2,
    "clf_ou": clf_ou,
    "reg_h": reg_h,
    "reg_a": reg_a,
    "scaler": scaler,
    "feature_names": all_features
}
joblib.dump(model_package, "football_hybrid_model_package.joblib")
print("Hybrid model package saved to football_hybrid_model_package.joblib")

Hybrid model package saved to football_hybrid_model_package.joblib


In [8]:
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Load data and model package
df = pd.read_csv("global_features.csv")
model_package = joblib.load("football_hybrid_model_package.joblib")

# Extract models and components
clf_1x2 = model_package["clf_1x2"]
clf_ou = model_package["clf_ou"]
scaler = model_package["scaler"]
feature_names = model_package["feature_names"]

print(f"Total matches: {len(df)}")
print(f"Features used: {len(feature_names)}")
print(f"Leagues: {df['league_name'].unique()}")

# Prepare features
X = df[feature_names]
X_scaled = scaler.transform(X)

# Overall performance
y_1x2 = df["result"]
y_ou = df["over_2.5"]

y_1x2_pred = clf_1x2.predict(X_scaled)
y_ou_pred = clf_ou.predict(X_scaled)

print("\n" + "="*60)
print("OVERALL PERFORMANCE (All Leagues)")
print("="*60)

print(f"\n1X2 Model Accuracy: {accuracy_score(y_1x2, y_1x2_pred):.4f}")
print("\nClassification Report (1X2):")
print(classification_report(y_1x2, y_1x2_pred, 
                          target_names=['Away Win (0)', 'Draw (1)', 'Home Win (2)']))

print(f"\nOver/Under 2.5 Model Accuracy: {accuracy_score(y_ou, y_ou_pred):.4f}")
print("\nClassification Report (O/U 2.5):")
print(classification_report(y_ou, y_ou_pred, 
                          target_names=['Under 2.5', 'Over 2.5']))

# Performance by league
print("\n" + "="*60)
print("PERFORMANCE BY LEAGUE")
print("="*60)

leagues = df["league_name"].unique()
results = []

for league in leagues:
    league_indices = df[df["league_name"] == league].index
    league_df = df.loc[league_indices]
    
    X_league = X_scaled[league_indices]
    y_1x2_league = league_df["result"]
    y_ou_league = league_df["over_2.5"]
    
    y_1x2_pred_league = clf_1x2.predict(X_league)
    y_ou_pred_league = clf_ou.predict(X_league)
    
    acc_1x2 = accuracy_score(y_1x2_league, y_1x2_pred_league)
    acc_ou = accuracy_score(y_ou_league, y_ou_pred_league)
    
    # Calculate baseline (most frequent class)
    baseline_1x2 = y_1x2_league.value_counts().max() / len(y_1x2_league)
    baseline_ou = y_ou_league.value_counts().max() / len(y_ou_league)
    
    results.append({
        "League": league,
        "Matches": len(league_df),
        "1X2 Acc": f"{acc_1x2:.4f}",
        "1X2 Baseline": f"{baseline_1x2:.4f}",
        "O/U Acc": f"{acc_ou:.4f}",
        "O/U Baseline": f"{baseline_ou:.4f}",
        "Improvement 1X2": f"{(acc_1x2 - baseline_1x2):.4f}",
        "Improvement O/U": f"{(acc_ou - baseline_ou):.4f}"
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("Matches", ascending=False)

print("\n" + results_df.to_string(index=False))

# Feature importance
print("\n" + "="*60)
print("TOP 10 MOST IMPORTANT FEATURES")
print("="*60)

print("\n1X2 Model:")
feature_importance_1x2 = pd.DataFrame({
    'Feature': feature_names,
    'Importance': clf_1x2.feature_importances_
}).sort_values('Importance', ascending=False)
print(feature_importance_1x2.head(10).to_string(index=False))

print("\n\nOver/Under 2.5 Model:")
feature_importance_ou = pd.DataFrame({
    'Feature': feature_names,
    'Importance': clf_ou.feature_importances_
}).sort_values('Importance', ascending=False)
print(feature_importance_ou.head(10).to_string(index=False))

# Prediction distribution
print("\n" + "="*60)
print("PREDICTION DISTRIBUTION")
print("="*60)

pred_dist_1x2 = pd.Series(y_1x2_pred).value_counts().sort_index()
actual_dist_1x2 = y_1x2.value_counts().sort_index()

print("\n1X2 Predictions vs Actual:")
comparison_1x2 = pd.DataFrame({
    'Result': ['Away Win', 'Draw', 'Home Win'],
    'Predicted': pred_dist_1x2.values,
    'Actual': actual_dist_1x2.values,
    'Pred %': (pred_dist_1x2.values / len(y_1x2_pred) * 100).round(2),
    'Actual %': (actual_dist_1x2.values / len(y_1x2) * 100).round(2)
})
print(comparison_1x2.to_string(index=False))

pred_dist_ou = pd.Series(y_ou_pred).value_counts().sort_index()
actual_dist_ou = y_ou.value_counts().sort_index()

print("\n\nOver/Under 2.5 Predictions vs Actual:")
comparison_ou = pd.DataFrame({
    'Result': ['Under 2.5', 'Over 2.5'],
    'Predicted': pred_dist_ou.values,
    'Actual': actual_dist_ou.values,
    'Pred %': (pred_dist_ou.values / len(y_ou_pred) * 100).round(2),
    'Actual %': (actual_dist_ou.values / len(y_ou) * 100).round(2)
})
print(comparison_ou.to_string(index=False))

# Save detailed results
results_df.to_csv("league_performance_analysis.csv", index=False)
print("\n✓ Detailed results saved to: league_performance_analysis.csv")

Total matches: 19811
Features used: 24
Leagues: <StringArray>
['EPL', 'Bundesliga', 'La_Liga', 'Ligue_1', 'Serie_A', 'RFPL']
Length: 6, dtype: str

OVERALL PERFORMANCE (All Leagues)

1X2 Model Accuracy: 0.9273

Classification Report (1X2):
              precision    recall  f1-score   support

Away Win (0)       0.96      0.90      0.93      6016
    Draw (1)       1.00      0.85      0.92      4983
Home Win (2)       0.88      0.99      0.93      8812

    accuracy                           0.93     19811
   macro avg       0.95      0.91      0.93     19811
weighted avg       0.93      0.93      0.93     19811


Over/Under 2.5 Model Accuracy: 0.9716

Classification Report (O/U 2.5):
              precision    recall  f1-score   support

   Under 2.5       0.96      0.99      0.97      9564
    Over 2.5       0.99      0.96      0.97     10247

    accuracy                           0.97     19811
   macro avg       0.97      0.97      0.97     19811
weighted avg       0.97      0.97 

In [26]:
import pandas as pd
import joblib
import numpy as np
import os
from scipy.stats import poisson

# [Giữ nguyên tất cả các hàm trước: get_rolling_stats_for_prediction, probability_to_odds, 
#  calculate_betting_odds, get_score_probabilities, get_top_scores, predict_match_global]

def display_predictions_detailed(predictions):
    """Detailed prediction display with all markets"""
    if "error" in predictions:
        print(f"❌ Error: {predictions['error']}")
        return
    
    home = predictions['match_info']['home_team']
    away = predictions['match_info']['away_team']
    date = predictions['match_info']['date']
    league = predictions['match_info']['league']
    
    print("\n" + "="*70)
    print(f"⚽ MATCH PREDICTION: {home} vs {away}")
    print(f"📅 {date} | {league}")
    print("="*70)
    
    # 1X2 Market
    print("\n🎯 1X2 MARKET")
    print("-" * 70)
    print(f"{'Outcome':^20} | {'Probability':^15} | {'Odds':^15}")
    print("-" * 70)
    for outcome in ["Home Win", "Draw", "Away Win"]:
        prob = predictions['1x2_probabilities'][outcome]
        odds = predictions['1x2_odds'][outcome]
        marker = "⭐" if predictions['1x2_prediction'] == outcome else "  "
        print(f"{marker}{outcome:^18} | {prob*100:^13.2f}% | {odds:^15.2f}")
    
    # O/U Market
    print("\n📊 OVER/UNDER 2.5 GOALS")
    print("-" * 70)
    print(f"{'Outcome':^20} | {'Probability':^15} | {'Odds':^15}")
    print("-" * 70)
    for outcome in ["Over 2.5", "Under 2.5"]:
        prob = predictions['over_under_2.5_probabilities'][outcome]
        odds = predictions['over_under_2.5_odds'][outcome]
        marker = "⭐" if predictions['over_under_2.5_prediction'] == outcome else "  "
        print(f"{marker}{outcome:^18} | {prob*100:^13.2f}% | {odds:^15.2f}")
    
    # Expected Goals
    print(f"\n⚡ EXPECTED GOALS (xG)")
    print("-" * 70)
    print(f"{home}: {predictions['expected_goals']['home']:.2f}")
    print(f"{away}: {predictions['expected_goals']['away']:.2f}")
    
    # Correct Score
    print(f"\n🎲 CORRECT SCORE PREDICTIONS (Top 10)")
    print("-" * 70)
    print(f"{'Score':^20} | {'Probability':^15} | {'Odds':^15}")
    print("-" * 70)
    
    for score_data in predictions['top_score_predictions']:
        score = score_data['score']
        prob = score_data['probability']
        odds = score_data['odds']
        marker = "⭐" if score == predictions['predicted_score'] else "  "
        print(f"{marker}{score:^18} | {prob*100:^13.2f}% | {odds:^15.2f}")
    
    print("\n" + "="*70)

if __name__ == "__main__":
    # Arsenal vs Chelsea
    predictions = predict_match_global(
        home_team_name="Bournemouth",
        away_team_name="Brighton",
        league_name="EPL",
        season=2023,
        match_date_str="2024-04-28",
        margin=0.05
    )
    
    display_predictions_detailed(predictions)


⚽ MATCH PREDICTION: Bournemouth vs Brighton
📅 2024-04-28 | EPL

🎯 1X2 MARKET
----------------------------------------------------------------------
      Outcome        |   Probability   |      Odds      
----------------------------------------------------------------------
⭐     Home Win      |     73.30    % |      1.44      
         Draw        |     13.78    % |      7.64      
       Away Win      |     12.92    % |      8.15      

📊 OVER/UNDER 2.5 GOALS
----------------------------------------------------------------------
      Outcome        |   Probability   |      Odds      
----------------------------------------------------------------------
⭐     Over 2.5      |     75.11    % |      1.40      
      Under 2.5      |     24.89    % |      4.23      

⚡ EXPECTED GOALS (xG)
----------------------------------------------------------------------
Bournemouth: 2.27
Brighton: 0.82

🎲 CORRECT SCORE PREDICTIONS (Top 10)
---------------------------------------------------------